In [22]:
import numpy as np, pandas as pd, json
from IPython.display import IFrame

RULES = ['Line of Fire','Energy Isolation','Work Authorisation','Mechanical Lifting',
         'Confined Space','Hot Work','Driving','Working at Height','Safe Mechanical Isolation']
RULES_COLORS = ['#2979FF','#FF9100','#00E676','#00BCD4','#FF4081','#FF1744','#B388FF','#76FF03','#FFD740']

SIF = ['Yes (SIF)', 'No (Non-SIF)']
SIF_COLORS = ['#FF1744', '#00B0FF']

In [23]:
# yahan real data wire karna — bas ye function replace karo
# df_rules mein columns: label, reports, color
# df_sif mein columns: label, reports, color

def generate_random_data(seed=None):
    rng = np.random.default_rng(seed)
    w_rules = np.clip(np.array([28,18,12,10,8,7,6,5,5],dtype=float)+rng.uniform(-4,4,9),1,40)
    t = rng.integers(2200,3800)
    r_rules = (w_rules/w_rules.sum()*t).astype(int); r_rules[-1]=t-r_rules[:-1].sum()
    df_rules = pd.DataFrame(dict(label=RULES, reports=r_rules, color=RULES_COLORS))
    
    w_sif = np.array([12, 88], dtype=float) + rng.uniform(-3, 3, 2)
    r_sif = (w_sif/w_sif.sum()*t).astype(int); r_sif[-1]=t-r_sif[:-1].sum()
    df_sif = pd.DataFrame(dict(label=SIF, reports=r_sif, color=SIF_COLORS))
    
    return df_rules, df_sif

df_rules, df_sif = generate_random_data()
total = int(df_rules.reports.sum())

def prepare_data(df):
    df = df.copy()
    df['pct'] = (df.reports/total*100).round(0).astype(int)
    return json.dumps([{'l':r.label,'v':int(r.reports),'p':int(r.pct),'c':r.color} for _,r in df.iterrows()]), df

dj_rules, df_rules = prepare_data(df_rules)
dj_sif, df_sif = prepare_data(df_sif)


In [24]:
def make_table(df, id_prefix):
    tr = ''
    for i, row in df.iterrows():
        tr += '<tr id="'+id_prefix+'-'+str(i)+'" style="transition:background .2s">'
        tr += '<td style="padding:7px 12px 7px 0;font-size:13px;white-space:nowrap">'
        tr += '<span style="display:inline-block;width:11px;height:11px;border-radius:50%;'
        tr += 'background:'+row.color+';margin-right:8px;vertical-align:middle"></span>'
        tr += row.label+'</td>'
        tr += '<td style="padding:7px 8px;text-align:right;font-size:13px;font-variant-numeric:tabular-nums">'+f"{row.reports:,}"+'</td>'
        tr += '<td style="padding:7px 0 7px 8px;text-align:right;font-size:13px">'+str(row.pct)+'%</td></tr>\n'
    tr += '<tr class="tr-t"><td style="padding:10px 12px 0 0;font-size:14px">Total</td>'
    tr += '<td style="padding:10px 8px 0;text-align:right;font-size:14px">'+f"{total:,}"+'</td>'
    tr += '<td style="padding:10px 0 0 8px;text-align:right;font-size:14px">100%</td></tr>'
    return tr

tbl_rules = make_table(df_rules, 'tr-rules')
tbl_sif = make_table(df_sif, 'tr-sif')

# css
css = ''
css += '* {margin:0;padding:0;box-sizing:border-box}'
css += 'body {background:#000;font-family:Inter,sans-serif;display:flex;align-items:center;justify-content:center;min-height:100vh}'
css += '#main {background:#000;padding:28px 36px 32px;border-radius:16px}'
css += '.tbar {display:flex;align-items:center;gap:14px;margin-bottom:18px}'
css += '.tbar h2 {font-size:20px;font-weight:700;color:#fff}'
css += '.tab {padding:6px 16px;border-radius:8px;font-size:13px;font-weight:600;cursor:pointer;transition:all .2s}'
css += '.tab-a {background:rgba(41,121,255,.15);color:#2979FF;border:1px solid rgba(41,121,255,.3)}'
css += '.tab-i {background:rgba(255,255,255,.05);color:#7b9ac8;border:1px solid rgba(100,150,255,.12)}'
css += '.tab-i:hover {background:rgba(255,255,255,.08);color:#a0b0d0}'
css += '.cnt {display:flex;align-items:center;gap:44px}'
css += '#cbox {position:relative;width:580px;height:480px;cursor:pointer}'
css += '#ci {position:absolute;text-align:center;pointer-events:none;transform:translate(-50%,-50%);z-index:1}'
css += '#cv {font-size:28px;font-weight:700;color:#fff;transition:color .25s}'
css += '#cl {font-size:13px;color:#7b9ac8;margin-top:2px;line-height:1.4}'
css += '.sd {color:#c0d0f0}'
css += '.sd h3 {font-size:15px;font-weight:600;color:#fff;margin-bottom:10px}'
css += '.sd table {border-collapse:collapse}'
css += '.sd th {padding:6px 12px 6px 0;text-align:left;color:#5a7aaa;font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:.5px;border-bottom:1px solid rgba(100,150,255,.25)}'
css += '.sd th:nth-child(2),.sd th:nth-child(3) {text-align:right;padding:6px 8px}'
css += '.sd td {border-bottom:1px solid rgba(100,150,255,.06)}'
css += '.tr-t {border-top:2px solid rgba(100,150,255,.3)!important}'
css += '.tr-t td {padding-top:10px!important;font-weight:700;color:#fff!important;font-size:14px!important}'
css += '#tt {position:fixed;display:none;pointer-events:none;background:rgba(10,14,30,0.95);color:#e0eaff;font-size:13px;padding:10px 14px;border-radius:6px;border:1px solid rgba(100,160,255,0.5);z-index:9999;box-shadow:0 4px 12px rgba(0,0,0,0.5);transform:translate(15px,15px)}'
css += '#tt b {color:#fff;font-size:14px;display:block;margin-bottom:6px}'
css += '.tt-dot {display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:6px;vertical-align:middle}'

# three.js code
js = ''
js += '(function(){'
js += 'var dataRules='+dj_rules+', dataSif='+dj_sif+';'
js += 'var total='+str(total)+', curData=dataSif, slices=[], hov=-1;'
js += 'var W=580,H=480,cv=document.getElementById("pc");'
js += 'var scene=new THREE.Scene();'
js += 'var cam=new THREE.PerspectiveCamera(32,W/H,.1,100);'
js += 'cam.position.set(0,2.8,3.8);cam.lookAt(0,.1,0);'
js += 'var ren=new THREE.WebGLRenderer({canvas:cv,antialias:true,alpha:true});'
js += 'ren.setSize(W,H);ren.setPixelRatio(Math.min(window.devicePixelRatio,2));'
js += 'scene.add(new THREE.AmbientLight(0xffffff,.45));'
js += 'var dl=new THREE.DirectionalLight(0xffffff,.9);dl.position.set(4,6,3);scene.add(dl);'
js += 'var d2=new THREE.DirectionalLight(0x4488ff,.2);d2.position.set(-3,4,-2);scene.add(d2);'
js += 'scene.add(new THREE.HemisphereLight(0x6688cc,0x111122,.25));'
js += 'var oR=1.3,iR=.54,bH=.28,gp=.025;'
js += 'function wdg(ir,or,sa,ea,h){'
js += 'var S=40,sh=new THREE.Shape(),da=(ea-sa)/S;'
js += 'sh.moveTo(or*Math.cos(sa),or*Math.sin(sa));'
js += 'for(var i=1;i<=S;i++)sh.lineTo(or*Math.cos(sa+da*i),or*Math.sin(sa+da*i));'
js += 'for(var i=S;i>=0;i--)sh.lineTo(ir*Math.cos(sa+da*i),ir*Math.sin(sa+da*i));'
js += 'var g=new THREE.ExtrudeGeometry(sh,{depth:h,bevelEnabled:true,bevelThickness:.012,bevelSize:.012,bevelSegments:3});'
js += 'g.rotateX(-Math.PI/2);return g;}'

js += 'function buildPie(data){'
js += '  for(var s of slices) { scene.remove(s.m); s.m.geometry.dispose(); s.m.material.dispose(); }'
js += '  slices=[]; hov=-1;'
js += '  var ang=Math.PI/2, N=data.length;'
js += '  for(var i=0;i<N;i++){'
js += '    var sw=(data[i].v/total)*Math.PI*2;'
js += '    var sa=ang+gp/2,ea=ang+sw-gp/2,mid=(sa+ea)/2;'
js += '    var g=wdg(iR,oR,sa,ea,bH);'
js += '    var m=new THREE.MeshPhongMaterial({color:new THREE.Color(data[i].c),shininess:80,specular:new THREE.Color(0x444444)});'
js += '    var ms=new THREE.Mesh(g,m); scene.add(ms);'
js += '    slices.push({m:ms,mid:mid,i:i,tY:0,tSY:1,tPX:0,tPZ:0});'
js += '    ang+=sw;'
js += '  }'
js += '}'
js += 'buildPie(curData);'

js += 'var ray=new THREE.Raycaster(),mp=new THREE.Vector2(-10,-10);'
js += 'var cV=document.getElementById("cv"),cL=document.getElementById("cl"),tt=document.getElementById("tt");'
js += 'var dV=cV.textContent,dL=cL.textContent,mX=0,mY=0;'
js += 'var cp=new THREE.Vector3(0,bH*.35,0).project(cam);'
js += 'var ci=document.getElementById("ci");'
js += 'ci.style.left=((cp.x+1)/2*100)+"%";ci.style.top=((-cp.y+1)/2*100)+"%";'

js += 'function hlR(x){'
js += '  var pfx=(curData===dataSif)?"tr-sif-":"tr-rules-";'
js += '  for(var j=0;j<curData.length;j++){'
js += '    var r=document.getElementById(pfx+j);'
js += '    if(r) r.style.background=(j===x)?"rgba(255,255,255,.08)":"";'
js += '  }'
js += '}'

js += 'cv.addEventListener("mousemove",function(e){var r=cv.getBoundingClientRect();mp.x=((e.clientX-r.left)/W)*2-1;mp.y=-((e.clientY-r.top)/H)*2+1;mX=e.clientX;mY=e.clientY;});'
js += 'cv.addEventListener("mouseleave",function(){mp.set(-10,-10);hov=-1;rst();});'
js += 'function rst(){for(var s of slices){s.tY=0;s.tSY=1;s.tPX=0;s.tPZ=0;s.m.material.emissive.set(0);}cV.textContent=dV;cV.style.color="#fff";cL.textContent=dL;cL.style.color="#7b9ac8";hlR(-1);tt.style.display="none";}'

js += 'var tRules=document.getElementById("t-rules"), tSif=document.getElementById("t-sif");'
js += 'var tbRules=document.getElementById("tb-rules"), tbSif=document.getElementById("tb-sif");'
js += 'var title=document.getElementById("sd-title");'
js += 'function setTab(isSif){'
js += '  rst();'
js += '  if(isSif){'
js += '    tSif.className="tab tab-a"; tRules.className="tab tab-i";'
js += '    tbSif.style.display=""; tbRules.style.display="none";'
js += '    curData=dataSif; title.textContent="SIF Classification";'
js += '  }else{'
js += '    tRules.className="tab tab-a"; tSif.className="tab tab-i";'
js += '    tbRules.style.display=""; tbSif.style.display="none";'
js += '    curData=dataRules; title.textContent="Life-Saving Rules";'
js += '  }'
js += '  buildPie(curData);'
js += '}'
js += 'tSif.onclick=function(){setTab(true);};'
js += 'tRules.onclick=function(){setTab(false);};'

js += 'function anim(){'
js += 'requestAnimationFrame(anim);'
js += 'ray.setFromCamera(mp,cam);'
js += 'var hits=ray.intersectObjects(slices.map(function(s){return s.m;}));'
js += 'var nh=-1;if(hits.length>0)for(var s of slices)if(s.m===hits[0].object){nh=s.i;break;}'
js += 'if(nh!==hov){hov=nh;'
js += 'if(hov>=0){var d=curData[hov];'
js += 'for(var s of slices){if(s.i===hov){s.tY=.10;s.tSY=1.45;s.tPX=Math.cos(s.mid)*.06;s.tPZ=-Math.sin(s.mid)*.06;s.m.material.emissive.set(0x151515);}'
js += 'else{s.tY=-.02;s.tSY=.82;s.tPX=0;s.tPZ=0;s.m.material.emissive.set(0);}}'
js += 'cV.textContent=d.v.toLocaleString();cV.style.color=d.c;'
js += 'cL.innerHTML=d.l+"<br><span style=\'font-size:11px;color:#7b9ac8\'>"+d.p+"% of total</span>";hlR(hov);'
js += 'tt.innerHTML="<b><span class=\'tt-dot\' style=\'background:"+d.c+"\'></span>"+d.l+"</b>Reports: "+d.v.toLocaleString()+"<br>Share: "+d.p+"%";tt.style.display="block";}'
js += 'else rst();}'
js += 'if(hov>=0){tt.style.left=mX+"px";tt.style.top=mY+"px";}'
js += 'var L=.10;for(var s of slices){s.m.position.x+=(s.tPX-s.m.position.x)*L;s.m.position.z+=(s.tPZ-s.m.position.z)*L;s.m.position.y+=(s.tY-s.m.position.y)*L;s.m.scale.y+=(s.tSY-s.m.scale.y)*L;}'
js += 'ren.render(scene,cam);}'
js += 'anim();})();'

# assemble html
h = '<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">'
h += '<title>SIF Classification</title>'
h += '<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap" rel="stylesheet">'
h += '<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"><' + '/script>'
h += '<style>'+css+'</style></head><body>'
h += '<div id="tt"></div>'
h += '<div id="main"><div class="tbar"><h2>Classification View</h2>'
h += '<span id="t-sif" class="tab tab-a">SIF Classification</span>'
h += '<span id="t-rules" class="tab tab-i">Life-Saving Rules ('+str(len(RULES))+')</span></div>'
h += '<div class="cnt"><div id="cbox"><canvas id="pc"></canvas>'
h += '<div id="ci"><div id="cv">'+f"{total:,}"+'</div><div id="cl">SIF Reports</div></div></div>'
h += '<div class="sd"><h3 id="sd-title">SIF Classification</h3>'
h += '<table id="tb-sif"><tr><th>Classification</th><th>Reports</th><th>%</th></tr>'+tbl_sif+'</table>'
h += '<table id="tb-rules" style="display:none"><tr><th>Rule</th><th>Reports</th><th>%</th></tr>'+tbl_rules+'</table>'
h += '</div></div></div>'
h += '<script>'+js+'<' + '/script></body></html>'

with open('sif_classification_3d_pie.html', 'w', encoding='utf-8') as f:
    f.write(h)

display(IFrame('sif_classification_3d_pie.html', width=1100, height=620))